In [1]:
from viewer.utils import load_env
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cmap

colormap = cmap.Colormap('tab20').to_matplotlib()
ks_output = Path(load_env('.env')["EPHYS_DATA_PATH"]) / "kilosort"

In [11]:
# LOAD PROBE CONTACTS 
ch_pos = np.load(ks_output / 'channel_positions.npy')
ch_shanks = np.load(ks_output / 'channel_shanks.npy').astype(int)
contacts_df = pd.DataFrame(ch_pos, columns=['x', 'y'])
contacts_df['shank'] = ch_shanks.astype(int)

# LOAD UNITS FROM KILOSORT OUTPUT
unit_labels = pd.read_csv(ks_output / 'cluster_group.tsv', sep='\t', index_col=0)
good_units = unit_labels[unit_labels['group'] == 'good'].index.values

clu = np.load(ks_output / 'spike_clusters.npy', mmap_mode='r')
spike_locations = np.load(ks_output / 'spike_positions.npy', mmap_mode='r')

good_mask = np.isin(clu, good_units)
spikes_df = pd.DataFrame(spike_locations[good_mask], columns=['x', 'y'])
spikes_df['unit_ids'] = clu[good_mask]
units_df = spikes_df.groupby('unit_ids').agg(['mean'])
units_df.columns = ['_'.join(col) for col in units_df.columns]

In [12]:
unit_xy = units_df[["x_mean", "y_mean"]].to_numpy(float)
contact_xy = contacts_df[["x", "y"]].to_numpy(float)

nearest_contact = np.argmin(
    ((unit_xy[:, None, :] - contact_xy[None, :, :]) ** 2).sum(axis=2),
    axis=1,
)
units_df["shank"] = contacts_df["shank"].to_numpy()[nearest_contact]
units_df

,x_mean,y_mean,shank
unit_ids,,,
18,12.412506,1007.089417,0
23,21.383923,1052.564941,0
26,13.031116,1065.637207,0
29,12.312404,1063.863647,0
32,11.823097,1076.413086,0
...,...,...,...
1063,261.985260,1021.237549,1
1064,265.271637,1021.180664,1
1077,269.083557,2080.519775,1


In [ ]:
shank_gap = 4.0
cursor = 0.0
units_df["unit_display_y"] = np.nan

for shank, group in units_df.groupby("shank", sort=True):
    order = group.sort_values(["y_mean", "x_mean"], ascending=[True, True]).index
    units_df.loc[order, "unit_display_y"] = cursor + np.arange(len(order), dtype=float)
    cursor += len(order) + shank_gap

units_df = units_df.sort_values("unit_display_y")
units_df

In [14]:
units_df.to_csv('units.csv')

In [3]:

units = pd.read_csv('./scripts/units.csv')
units['unit_ids']

0        18
1        23
2        29
3        26
4        32
       ... 
301    1002
302    1004
303    1006
304    1019
305    1037
Name: unit_ids, Length: 306, dtype: int64

In [9]:
from viewer.utils import filter_units

filter_units(
    spike_times=np.load(ks_output / 'spike_times.npy'),
    spike_units=np.load(ks_output / 'spike_clusters.npy'),
    unit_ids=units['unit_ids'].values,
    output_path='.',
    unit_ids_filename=None
)

{'paths': {'spike_times': WindowsPath('spike_times.npy'),
  'spike_units': WindowsPath('spike_units.npy')},
 'n_spikes': 43752088,
 'n_units': 306}

In [11]:
np.unique(np.load('spike_units.npy'))

array([  18,   23,   26,   29,   32,   38,   44,   46,   47,   54,   56,
         57,   58,   71,   75,   77,   79,   83,   84,   86,   93,   94,
         99,  100,  106,  107,  112,  115,  117,  119,  124,  127,  128,
        132,  133,  134,  138,  139,  140,  142,  143,  146,  147,  149,
        151,  152,  153,  154,  157,  160,  166,  167,  169,  170,  171,
        172,  177,  180,  181,  185,  189,  192,  193,  196,  197,  198,
        200,  201,  203,  206,  208,  214,  217,  219,  220,  222,  225,
        227,  228,  229,  231,  232,  235,  237,  238,  240,  245,  248,
        249,  251,  252,  253,  258,  263,  266,  272,  280,  285,  287,
        291,  292,  294,  299,  300,  301,  307,  311,  313,  319,  328,
        329,  339,  340,  348,  350,  351,  353,  354,  366,  370,  375,
        376,  382,  383,  384,  385,  392,  393,  396,  398,  401,  402,
        403,  406,  413,  418,  419,  424,  426,  431,  432,  436,  437,
        442,  451,  453,  454,  457,  462,  465,  4